In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time, os, glob

spark = (SparkSession.builder
         .appName("hafta5")
         .master("local[4]")
         .getOrCreate())

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

KOK = "work/gol" if os.path.isdir("work/gol") else "gol"

gol = (spark.read
       .option("basePath", KOK)
       .parquet(KOK + "/yil=*"))

gol.createOrReplaceTempView("yorumlar")
print("Hazir. Parca sayisi:", gol.rdd.getNumPartitions())

Hazir. Parca sayisi: 31


In [3]:
def olc(etiket, fonksiyon):
    t0 = time.time()
    sonuc = fonksiyon()
    sure = time.time() - t0
    print(f"{etiket:<34} {sure:7.2f} sn")
    return sure


In [4]:
olc("ornek sorgu", lambda: spark.sql("SELECT COUNT(*) FROM yorumlar").collect())

ornek sorgu                           6.11 sn


6.105342149734497

In [5]:
gol.printSchema()

root
 |-- rating: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- text: string (nullable = true)
 |-- zaman: long (nullable = true)
 |-- yil: integer (nullable = true)



In [6]:
a = olc("Sadece puan sutunu",
        lambda: spark.sql("SELECT AVG(rating) FROM yorumlar").collect())
 
b = olc("Metin sutunu da dahil",
        lambda: spark.sql("SELECT AVG(LENGTH(text)) FROM yorumlar").collect())
 
print(f"\nIkincisi {b/a:.1f} kat uzun surdu")


Sadece puan sutunu                    5.94 sn
Metin sutunu da dahil                 8.60 sn

Ikincisi 1.4 kat uzun surdu


In [8]:
spark.sql("SELECT AVG(LENGTH(text)) FROM yorumlar").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[avg(length(text#6))])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=174]
      +- HashAggregate(keys=[], functions=[partial_avg(length(text#6))])
         +- Project [text#6]
            +- FileScan parquet [text#6,yil#8] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(24 paths)[file:/home/jovyan/work/gol/yil=2000, file:/home/jovyan/work/gol/yil=2..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<text:string>




In [9]:
olc("Gruplama yok",
    lambda: spark.sql("SELECT COUNT(*) FROM yorumlar WHERE yil = 2020").collect())
 
olc("Gruplama var",
    lambda: spark.sql("""SELECT parent_asin, COUNT(*)
                        FROM yorumlar GROUP BY parent_asin""").collect())


Gruplama yok                          0.44 sn
Gruplama var                          7.93 sn


7.934812784194946

In [10]:
olc("Ayni sutun, gruplama yok",
    lambda: spark.sql("""SELECT COUNT(*) FROM yorumlar
                        WHERE parent_asin IS NOT NULL""").collect())


Ayni sutun, gruplama yok              6.82 sn


6.82047963142395

In [11]:
print(spark.conf.get("spark.sql.shuffle.partitions"))

200


In [12]:
print("Otomatik ayar acik mi:", spark.conf.get("spark.sql.adaptive.enabled"))

Otomatik ayar acik mi: true


In [13]:
SORGU = """SELECT parent_asin, COUNT(*) AS adet, AVG(rating) AS ort
           FROM yorumlar GROUP BY parent_asin"""
 	
spark.conf.set("spark.sql.adaptive.enabled", "false")
 
for n in ["200", "16", "4"]:
    spark.conf.set("spark.sql.shuffle.partitions", n)
    olc(f"bolme sayisi = {n}",
        lambda: spark.sql(SORGU).collect())
 
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("\nOtomatik ayar tekrar acildi")


bolme sayisi = 200                    9.64 sn
bolme sayisi = 16                     6.41 sn
bolme sayisi = 4                      6.39 sn

Otomatik ayar tekrar acildi
